# Method 4: Canny + Sequential RANSAC
Uses Canny edge detection (instead of skeletonization) as input to Sequential RANSAC with gap detection.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.measure import ransac, LineModelND

img_color = cv2.imread('cuboid.png')
img_rgb = cv2.cvtColor(img_color, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(img_color, cv2.COLOR_BGR2GRAY)
blurred = cv2.GaussianBlur(gray, (5, 5), 1.5)
edges = cv2.Canny(blurred, 50, 150)

# Extract edge pixel coordinates
ys, xs = np.nonzero(edges)
points = np.column_stack((xs, ys)).astype(np.float64)
print(f'Edge pixels: {len(points)}')

# Sequential RANSAC
detected_lines = []
remaining = points.copy()
residual_threshold = 2.0  # Canny produces double edges, so slightly wider threshold
max_gap = 15

for i in range(15):
    if len(remaining) < 50:
        break
    model, inlier_mask = ransac(remaining, LineModelND, min_samples=2,
                                residual_threshold=residual_threshold, max_trials=1000)
    inliers = remaining[inlier_mask]
    if len(inliers) < 30:
        break

    # Gap detection — keep largest continuous chunk
    x_range = inliers[:, 0].max() - inliers[:, 0].min()
    y_range = inliers[:, 1].max() - inliers[:, 1].min()
    sort_idx = np.argsort(inliers[:, 0] if x_range >= y_range else inliers[:, 1])
    sorted_pts = inliers[sort_idx]
    diffs = np.linalg.norm(np.diff(sorted_pts, axis=0), axis=1)
    gap_pos = np.where(diffs > max_gap)[0]
    splits = np.concatenate(([0], gap_pos + 1, [len(sorted_pts)]))
    chunks = [sorted_pts[splits[j]:splits[j+1]] for j in range(len(splits)-1)]
    best = max(chunks, key=len)
    if len(best) < 30:
        break

    origin, direction = model.origin, model.direction
    proj = (best - origin) @ direction
    p0 = origin + proj.min() * direction
    p1 = origin + proj.max() * direction
    detected_lines.append((p0[0], p0[1], p1[0], p1[1]))
    print(f'Line {i+1}: ({p0[0]:.0f},{p0[1]:.0f})->({p1[0]:.0f},{p1[1]:.0f}) inliers:{len(inliers)} chunk:{len(best)}')
    remaining = remaining[~inlier_mask]

# Plot
fig, axes = plt.subplots(1, 2, figsize=(20, 8))
axes[0].imshow(img_rgb); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(img_rgb)
axes[1].set_title(f'Canny + RANSAC ({len(detected_lines)} segments)')
axes[1].axis('off')
for x0, y0, x1, y1 in detected_lines:
    axes[1].plot([x0, x1], [y0, y1], color='lime', linewidth=2.5)
plt.tight_layout()
fig.savefig('result_canny_ransac.png', dpi=150, bbox_inches='tight')
plt.show()